Extract data from csv

In [640]:
import pandas as pd
import numpy as np

In [641]:
df = pd.read_csv('data/bronze/dirty_cafe_sales.csv')

In [642]:
df.sample(5)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
4781,TXN_8170122,Cookie,5,1.0,5.0,Digital Wallet,In-store,2023-03-20
7451,TXN_2377667,Smoothie,3,4.0,12.0,Digital Wallet,Takeaway,2023-10-27
8458,TXN_7595907,NaN,5,ERROR,15.0,NaN,In-store,2023-04-24
9845,TXN_1928401,Sandwich,5,4.0,20.0,Digital Wallet,In-store,2023-06-15
5790,TXN_3020475,Salad,4,5.0,20.0,NaN,Takeaway,2023-09-13


In [643]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [644]:
df_copy = df.copy()

In [645]:
df_copy.columns = [c.lower().replace(' ', '_') for c in df_copy.columns]

In [646]:
df_copy.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

In [647]:
df_copy['item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

In [648]:
def is_missing_flag(df, columns, value): 
    for column in columns:
        df.loc[(df[column] == value) | (df[column].isna()), 'is_missing'] = True

In [649]:
def is_error_flag(df, columns, value): 
    for column in columns:
        df.loc[df_copy[column] == value, 'is_error'] = True

In [650]:
def define_flags(df):
   df['is_error'] = False

   df['is_missing'] = False

   columns_for_missing_flag = ['item', 'quantity', 'price_per_unit', 'total_spent', 'transaction_date']
   is_missing_flag(df, columns_for_missing_flag, 'UNKNOWN')

   columns_for_error_flag = ['item', 'quantity', 'price_per_unit', 'total_spent']
   is_error_flag(df, columns_for_error_flag, 'ERROR')

   return df

In [651]:
df_copy = define_flags(df_copy)

In [652]:
df_copy[df_copy['is_missing'] == True]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
25,TXN_7958992,Smoothie,3,4.0,NaN,UNKNOWN,UNKNOWN,2023-12-13,False,True
30,TXN_1736287,NaN,5,2.0,10.0,Digital Wallet,NaN,2023-06-02,False,True
31,TXN_8927252,UNKNOWN,2,1.0,ERROR,Credit Card,ERROR,2023-11-06,True,True
...,...,...,...,...,...,...,...,...,...,...
9988,TXN_9594133,Cake,5,3.0,NaN,ERROR,NaN,NaN,False,True
9993,TXN_4766549,Smoothie,2,4.0,NaN,Cash,NaN,2023-10-20,False,True
9994,TXN_7851634,UNKNOWN,4,4.0,16.0,NaN,NaN,2023-01-08,False,True
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02,False,True


In [653]:
df_copy.replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)

In [654]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    10000 non-null  object
 1   item              9031 non-null   object
 2   quantity          9521 non-null   object
 3   price_per_unit    9467 non-null   object
 4   total_spent       9498 non-null   object
 5   payment_method    6822 non-null   object
 6   location          6039 non-null   object
 7   transaction_date  9540 non-null   object
 8   is_error          10000 non-null  bool  
 9   is_missing        10000 non-null  bool  
dtypes: bool(2), object(8)
memory usage: 644.7+ KB


In [655]:
df_copy['price_per_unit'] = df_copy['price_per_unit'].astype(float)

In [656]:
df_copy['total_spent'] = df_copy['total_spent'].astype(float)

In [657]:
df_copy['quantity'] = pd.to_numeric(df_copy['quantity'], errors='coerce').astype('Int64')

In [658]:
df_copy['transaction_date'] = pd.to_datetime(df_copy['transaction_date'])

In [659]:
type(df_copy['quantity'][0])

numpy.int64

In [660]:
df_copy.loc[
   (df_copy['price_per_unit'].isna() &
    df_copy['total_spent'].notna() &
    df_copy['quantity'].notna()
    ), 'price_per_unit'] = df_copy['total_spent']/df_copy['quantity']

In [661]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9031 non-null   object        
 2   quantity          9521 non-null   Int64         
 3   price_per_unit    9962 non-null   float64       
 4   total_spent       9498 non-null   float64       
 5   payment_method    6822 non-null   object        
 6   location          6039 non-null   object        
 7   transaction_date  9540 non-null   datetime64[ns]
 8   is_error          10000 non-null  bool          
 9   is_missing        10000 non-null  bool          
dtypes: Int64(1), bool(2), datetime64[ns](1), float64(2), object(4)
memory usage: 654.4+ KB


In [662]:
df_copy['item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', nan, 'Sandwich',
       'Juice', 'Tea'], dtype=object)

In [663]:
df_copy[df_copy['item'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,NaN,3,3.0,9.0,NaN,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
14,TXN_8915701,NaN,2,1.5,3.0,NaN,In-store,2023-03-21,True,False
30,TXN_1736287,NaN,5,2.0,10.0,Digital Wallet,NaN,2023-06-02,False,True
31,TXN_8927252,NaN,2,1.0,NaN,Credit Card,NaN,2023-11-06,True,True
...,...,...,...,...,...,...,...,...,...,...
9951,TXN_4122925,NaN,4,1.0,4.0,NaN,Takeaway,2023-10-20,True,False
9958,TXN_4125474,NaN,2,5.0,10.0,Credit Card,In-store,2023-08-02,True,False
9981,TXN_4583012,NaN,5,4.0,20.0,Digital Wallet,NaN,2023-02-27,True,False
9994,TXN_7851634,NaN,4,4.0,16.0,NaN,NaN,2023-01-08,False,True


In [664]:
mapping_df = df_copy.loc[
   (df_copy['item'].notna()) & 
   (df_copy['price_per_unit'].notna()),
   ['item', 'price_per_unit']].drop_duplicates()

In [665]:
mapping_price_dict = dict(zip(mapping_df['item'], mapping_df['price_per_unit']))

In [666]:
mapping_price_dict

{'Coffee': 2.0,
 'Cake': 3.0,
 'Cookie': 1.0,
 'Salad': 5.0,
 'Smoothie': 4.0,
 'Sandwich': 4.0,
 'Juice': 3.0,
 'Tea': 1.5}

In [667]:
mapping_df.drop_duplicates(subset=['price_per_unit'], keep=False, inplace=True)

In [668]:
mapping_item_dict = dict(zip(mapping_df['item'], mapping_df['price_per_unit']))

In [669]:
mapping_item_dict

{'Coffee': 2.0, 'Cookie': 1.0, 'Salad': 5.0, 'Tea': 1.5}

In [670]:
for item, price in mapping_item_dict.items():
   mask = (df_copy['item'].isna()) & (df_copy['price_per_unit'] == price)
   df_copy.loc[mask, 'item'] = item

In [671]:
df_copy[df_copy['item'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,NaN,3,3.0,9.0,NaN,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
36,TXN_6855453,NaN,4,3.0,12.0,NaN,In-store,2023-07-17,False,True
61,TXN_8051289,NaN,1,3.0,3.0,NaN,In-store,2023-10-09,False,True
69,TXN_8471743,NaN,5,3.0,15.0,Digital Wallet,In-store,2023-04-06,True,False
...,...,...,...,...,...,...,...,...,...,...
9910,TXN_2338617,NaN,2,3.0,6.0,Digital Wallet,NaN,2023-01-12,True,False
9918,TXN_2292088,NaN,1,4.0,4.0,Digital Wallet,Takeaway,2023-03-04,True,False
9946,TXN_8807600,NaN,1,4.0,4.0,Cash,Takeaway,2023-09-24,False,True
9981,TXN_4583012,NaN,5,4.0,20.0,Digital Wallet,NaN,2023-02-27,True,False


In [672]:
for item, price in mapping_price_dict.items():
   mask = (df_copy['price_per_unit'].isna()) & (df_copy['item'] == item)
   df_copy.loc[mask, 'price_per_unit'] = price

In [673]:
df_copy.loc[
   (df_copy['price_per_unit'].notna()) &
   (df_copy['total_spent'].isna()) &
   (df_copy['quantity'].notna()
   ), 'total_spent'] = df_copy['price_per_unit'] * df_copy['quantity']

In [674]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9520 non-null   object        
 2   quantity          9521 non-null   Int64         
 3   price_per_unit    9994 non-null   float64       
 4   total_spent       9977 non-null   float64       
 5   payment_method    6822 non-null   object        
 6   location          6039 non-null   object        
 7   transaction_date  9540 non-null   datetime64[ns]
 8   is_error          10000 non-null  bool          
 9   is_missing        10000 non-null  bool          
dtypes: Int64(1), bool(2), datetime64[ns](1), float64(2), object(4)
memory usage: 654.4+ KB


In [675]:
df_copy[df_copy['total_spent'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
236,TXN_8562645,Salad,<NA>,5.0,NaN,NaN,In-store,2023-05-18,True,True
278,TXN_3229409,Juice,<NA>,3.0,NaN,Cash,Takeaway,2023-04-15,False,True
641,TXN_2962976,Juice,<NA>,3.0,NaN,NaN,NaN,2023-03-17,True,True
738,TXN_8696094,Sandwich,<NA>,4.0,NaN,NaN,Takeaway,2023-05-14,False,True
1761,TXN_3611851,NaN,4,NaN,NaN,Credit Card,NaN,2023-02-09,True,True
2289,TXN_7524977,NaN,4,NaN,NaN,NaN,NaN,2023-12-09,False,True
2796,TXN_9188692,Cake,<NA>,3.0,NaN,Credit Card,NaN,2023-12-01,True,True
3203,TXN_4565754,Smoothie,<NA>,4.0,NaN,Digital Wallet,Takeaway,2023-10-06,True,True
3224,TXN_6297232,Coffee,<NA>,2.0,NaN,NaN,NaN,2023-04-07,False,True
3401,TXN_3251829,Tea,<NA>,1.5,NaN,Digital Wallet,In-store,2023-07-25,False,True


In [ ]:
df_copy.to_parquet('data/silver/cleaned_cafe_sales.parquet')

ArrowKeyError: A type extension with name pandas.period already defined